In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Dataset - Direct Link (Download ka masla khatam)
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', names=['label','message'])
print("Data Loaded:", df.shape)
print(df.head())

# 2. Simple Cleaning (No NLTK needed)
def clean_text(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    return text

df['clean_message'] = df['message'].apply(clean_text)

# 3. TF-IDF
vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')
X = vectorizer.fit_transform(df['clean_message'])
y = df['label'].map({'ham':0, 'spam':1})

# 4. Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = MultinomialNB()
model.fit(X_train, y_train)

# 5. Accuracy
y_pred = model.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 6. Prediction Function
def predict_email(text):
    cleaned = clean_text(text)
    vect = vectorizer.transform([cleaned])
    pred = model.predict(vect)[0]
    prob = max(model.predict_proba(vect)[0])*100
    result = "SPAM" if pred==1 else "HAM (Not Spam)"
    return result, prob

# Test
print("\n--- Test ---")
print(predict_email("Congratulations you won $1000 prize click now"))
print(predict_email("Hi Ansa, meeting tomorrow at 10am?"))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Spam vs Ham Distribution Graph
plt.figure(figsize=(6,4))
sns.countplot(x=df['label'], palette='viridis')
plt.title('Spam vs Ham Distribution (5572 Emails)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

# 2. Confusion Matrix Graph
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', xticklabels=['Ham','Spam'], yticklabels=['Ham','Spam'])
plt.title('Confusion Matrix - 98.48% Accuracy')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# 3. WordCloud (Optional)
from wordcloud import WordCloud
spam_text = " ".join(df[df['label']=='spam']['message'])
wordcloud = WordCloud(width=800, height=400, background_color='black').generate(spam_text)
plt.figure(figsize=(10,5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in SPAM Emails')
plt.show()